Notebook pre-requisites:

In [156]:
!pip install "camelot-py[cv]"
!pip install spacy



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [157]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.2 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [158]:
!pip install PyMuPDF
!pip install pdfplumber


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 1. PDF Ingestion & Parsing
- Extract text with page/section anchors (page number, heading hierarchy).
- Preserve structure: titles, subsections, lists, tables, figures’ captions.
- For tables: parse into machine-readable frames (CSV/JSON) when possible.
- Deliverables: raw_text.jsonl (chunks with metadata), tables/*.csv.

In [159]:
import fitz
import json
import re
from pathlib import Path
import shutil
import camelot
import pdfplumber
import pandas as pd

# =============== TOC EXTRACTION ===============
def extract_contents_section(doc):
    toc_lines = []
    toc_started = False
    current_title = ""
    suppressing = False  # True when we've seen THL but not yet seen FOOD

    toc_header_re = re.compile(r"\b(Contents|Table of Contents)\b", re.IGNORECASE)
    toc_entry_re = re.compile(r"\.{3,}\s*(\d+)$")  # dots then page number at EOL
    thl_re = re.compile(r"THL", re.IGNORECASE)
    food_re = re.compile(r"FOOD", re.IGNORECASE)

    for page in doc:
        lines = page.get_text().splitlines()
        for line in lines:
            stripped = line.strip()
            if not stripped:
                continue
            if not toc_started and toc_header_re.search(stripped):
                toc_started = True
                continue
            if not toc_started:
                continue

            # suppression logic
            if suppressing:
                m_food = food_re.search(stripped)
                if m_food:
                    stripped = stripped[m_food.end():].strip()
                    suppressing = False
                    if not stripped:
                        continue
                else:
                    continue

            m_thl = thl_re.search(stripped)
            if m_thl:
                m_food = food_re.search(stripped, m_thl.end())
                if m_food:
                    stripped = stripped[m_food.end():].strip()
                    if not stripped:
                        continue
                else:
                    stripped = stripped[:m_thl.start()].strip()
                    suppressing = True
                    if not stripped:
                        continue

            match = toc_entry_re.search(stripped)
            if match:
                full_title = (current_title + " " + stripped).strip() if current_title else stripped
                toc_lines.append(full_title)
                current_title = ""
            else:
                if current_title:
                    current_title += " " + stripped
                else:
                    current_title = stripped

        last_few = lines[-5:]
        if toc_started and all(not toc_entry_re.search(l.strip()) for l in last_few):
            break

    if current_title:
        if suppressing:
            last_thl = re.search(r"THL", current_title, re.IGNORECASE)
            cleaned_title = current_title[:last_thl.start()].strip() if last_thl else ""
        else:
            cleaned_title = current_title.strip()
        if cleaned_title:
            toc_lines.append(cleaned_title)

    return "\n".join(toc_lines)

# =============== PARSE CONTENTS TO DATAFRAME ===============
def parse_contents_to_df(contents_text):
    lines = [l.strip() for l in contents_text.split("\n") if re.search(r"\d+\s*$", l)]
    rows = []
    for line in lines:
        match = re.match(r"(.+?)\s+(\d+)$", line)
        if match:
            title, page = match.groups()
            title = re.sub(r"\.{2,}", "", title).strip()
            rows.append([title, int(page)])
    df = pd.DataFrame(rows, columns=["title", "page"])
    df = merge_split_rows(df)
    return df

def merge_split_rows(df):
    return df

def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip().lower()

# =============== TABLE EXTRACTION USING CAMELT/PLUMBER ===============
def extract_tables(pdf_path, output_dir):
    output_dir = Path(output_dir)
    tables_dir = output_dir / "tables"
    if tables_dir.exists():
        shutil.rmtree(tables_dir)
    tables_dir.mkdir(exist_ok=True)

    all_tables = []
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page_str = str(page_num + 1)
        page_tables = []

        # Camelot STREAM
        try:
            tables_stream = camelot.read_pdf(pdf_path, pages=page_str, flavor='stream', edge_tol=50, row_tol=10, strip_text='\n')
            page_tables += [t.df for t in tables_stream if not t.df.empty]
        except Exception:
            pass

        # Camelot LATTICE
        if not page_tables:
            try:
                tables_lattice = camelot.read_pdf(pdf_path, pages=page_str, flavor='lattice', line_scale=40, shift_text=['l','t'])
                page_tables += [t.df for t in tables_lattice if not t.df.empty]
            except Exception:
                pass

        # pdfplumber fallback
        if not page_tables:
            with pdfplumber.open(pdf_path) as pdf:
                page = pdf.pages[page_num]
                plumber_tables = page.extract_tables()
                for pt in plumber_tables:
                    df = pd.DataFrame(pt)
                    page_tables.append(df)

        # Save tables
        for i, df in enumerate(page_tables):
            df = df.fillna("").astype(str).apply(lambda col: col.map(lambda x: re.sub(r"\n", " ", x).strip()))
            df = merge_split_rows(df)
            table_file = tables_dir / f"table_page{page_num+1}_{i+1}.csv"
            df.to_csv(table_file, index=False)
            all_tables.append({
                "table_number": i + 1,
                "page": page_num + 1,
                "file": str(table_file),
                "rows": df.shape[0],
                "columns": df.shape[1]
            })

    print(f"✓ Extracted {len(all_tables)} tables total to '{tables_dir}'")
    return all_tables

# =============== MAIN EXTRACTION FUNCTION ===============
def extract_pdf_with_structure(pdf_path, output_dir="data"):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    doc = fitz.open(pdf_path)
    chunks = []

    # --- Extract TOC and parse ---
    contents_text = extract_contents_section(doc)
    toc_df = parse_contents_to_df(contents_text)
    toc_df = toc_df.sort_values(by="page").reset_index(drop=True)

    toc_index = 0
    current_section = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        lines = page.get_text().splitlines()

        new_sections = []
        while toc_index < len(toc_df) and toc_df.loc[toc_index, "page"] <= page_num + 1:
            new_sections.append(toc_df.loc[toc_index, "title"])
            toc_index += 1

        pending_sections = new_sections.copy()
        buffer = []
        line_idx = 0

        while line_idx < len(lines):
            matched_section = None
            matched_lines_count = 0
            for section_title in pending_sections:
                for lookahead in range(1, min(5, len(lines) - line_idx + 1)):
                    candidate_text = " ".join(lines[line_idx:line_idx+lookahead])
                    if normalize_text(candidate_text).startswith(normalize_text(section_title)):
                        matched_section = section_title
                        matched_lines_count = lookahead
                        break
                if matched_section:
                    break

            if matched_section:
                if current_section and buffer:
                    text_chunk = " ".join(buffer).strip()
                    if text_chunk:
                        chunks.append({"page": page_num+1, "section": current_section, "text": text_chunk})
                current_section = matched_section
                buffer = []
                pending_sections.remove(matched_section)
                line_idx += matched_lines_count
                continue
            else:
                if current_section:
                    buffer.append(lines[line_idx].strip())
                line_idx += 1

        if current_section and buffer:
            text_chunk = " ".join(buffer).strip()
            if text_chunk:
                chunks.append({"page": page_num+1, "section": current_section, "text": text_chunk})

        if pending_sections:
            for missing_section in pending_sections:
                print(f"[WARNING] Section '{missing_section}' expected on page {page_num+1} but not found.")
                if buffer:
                    text_chunk = " ".join(buffer).strip()
                    if text_chunk:
                        chunks.append({"page": page_num+1, "section": missing_section, "text": text_chunk})
                        buffer = []

    # Save text
    output_file = output_dir / "raw_text.jsonl"
    with open(output_file, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print(f"✓ Extracted {len(chunks)} text chunks")
    print(f"✓ Clean text saved to {output_file}")

    # --- Extract tables using Camelot/pdfplumber ---
    tables_info = extract_tables(pdf_path, output_dir)

    return chunks, toc_df, tables_info


In [160]:
pdf_path = "data/sustainable-health-from-food_web.pdf"
chunks, toc_df, tables = extract_pdf_with_structure(pdf_path)

[WARNING] Section 'Appendix 8. Recommended intakes of fat, carbohydrates and protein for adults and children over 2 years (without alcohol and with fibre taken into account)' expected on page 102 but not found.
✓ Extracted 134 text chunks
✓ Clean text saved to data/raw_text.jsonl


/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (0, 0, 498.898, 708.661)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (63.7008, 137.1649, 437.67889999999966, 681.5164555555556)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (92.0472, 302.90479999999997, 462.89330000000035, 684.5557947368421)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.

✓ Extracted 137 tables total to 'data/tables'


/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (53.76729999999999, 246.3673, 449.4282999999997, 630.1827161904762)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


## 3) NER & Keyphrase Extraction

In [161]:
!pip install spacy
!pip install transformers
!pip install keybert
!pip install scispacy
!pip install fuzzywuzzy python-Levenshtein
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
  Using cached scispacy-0.6.2-py3-none-any.whl.metadata (20 kB)
  Using cached conllu-6.0.0-py3-none-any.whl.metadata (21 kB)
  Using cached numpy-1.26.4-cp310-cp310-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached nmslib-metabrainz-2.1.3.tar.gz (196 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pysbd-0.3.4-py3-none-any.whl.metadata (6.1 kB)
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
INFO: pip is looking at multiple versions of thinc to determine which version is compatible with other requ

In [162]:
import json
import pandas as pd
import spacy
from keybert import KeyBERT
import yaml

# load Text Chunks
with open('data/raw_text.jsonl', 'r') as f:
    text_chunks = [json.loads(line) for line in f]
df_chunks = pd.DataFrame(text_chunks)
print(f"Loaded {len(df_chunks)} text chunks.")

# Load Ontology
with open('data/ontology.yaml', 'r') as f:
    ontology = yaml.safe_load(f)
entity_classes = list(ontology.get('classes', {}).keys())
print(f"Ontology classes: {entity_classes}")

Loaded 134 text chunks.
Ontology classes: ['ingredient', 'nutrient', 'technique', 'dietaryGuideline', 'healthOutcome', 'environmentImpact']


In [163]:
!pip install yake


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 4. Relation Extraction & Triple Building
- Use rule-based patterns and/or relation extraction models to detect relations (e.g., Ingredient X contains
Nutrient Y, Technique Z requires Temperature T).
- Convert to triples (RDF or property graph). Include provenance (page, line span).
- Deduplicate and validate (schema consistency).
- Deliverables: triples.ttl (RDF) or graph.json (property graph).

In [164]:
import json
from rdflib import Graph, Namespace, URIRef, Literal
from collections import defaultdict

# Load ontology namespace
EX = Namespace("http://example.org/food#")

# Initialize RDF graph
g = Graph()
g.bind("ex", EX)

# Load entities
entities = []
with open("data/entities_lowercased.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        entities.append(json.loads(line))

# Flatten mentions: one record per page + entity
flat_entities = []
for e in entities:
    for mention in e.get("mentions", []):
        flat_entities.append({
            "id": e["id"],
            "entity": e["label"],
            "ontology_type": e["type"],
            "page": mention["page"],
            "surface": mention["surface"],
            "context": mention.get("context", "")
        })

# Simple canonicalization: lowercasing and deduplicating by page & type
canonical_entities = {}
for e in flat_entities:
    key = (e["page"], e["ontology_type"], e["entity"].lower())
    canonical_entities[key] = e

# Group entities by page
entities_by_page = defaultdict(list)
for (page, otype, _), e in canonical_entities.items():
    entities_by_page[page].append(e)

# Relation extraction rules (simple co-occurrence + keyword heuristics)
def extract_relations(page_entities):
    triples = []
    for e1 in page_entities:
        for e2 in page_entities:
            if e1 == e2:
                continue
            # Ingredient → Nutrient
            if e1["ontology_type"] == "ingredient" and e2["ontology_type"] == "nutrient":
                triples.append((e1["entity"], "hasNutrient", e2["entity"]))

            # Ingredient → Technique
            elif e1["ontology_type"] == "ingredient" and e2["ontology_type"] == "technique":
                triples.append((e1["entity"], "usesTechnique", e2["entity"]))

            # Ingredient → DietaryGuideline
            elif e1["ontology_type"] == "ingredient" and e2["ontology_type"] == "dietaryGuideline":
                triples.append((e1["entity"], "hasGuideline", e2["entity"]))

            # Ingredient → HealthOutcome
            elif e1["ontology_type"] == "ingredient" and e2["ontology_type"] == "healthOutcome":
                triples.append((e1["entity"], "associatedWithOutcome", e2["entity"]))

            # Ingredient → EnvironmentImpact
            elif e1["ontology_type"] == "ingredient" and e2["ontology_type"] == "environmentImpact":
                triples.append((e1["entity"], "hasEnvironmentalImpact", e2["entity"]))

            # Nutrient → HealthOutcome
            elif e1["ontology_type"] == "nutrient" and e2["ontology_type"] == "healthOutcome":
                triples.append((e1["entity"], "affectsRiskOf", e2["entity"]))

            # DietaryGuideline → Technique
            elif e1["ontology_type"] == "dietaryGuideline" and e2["ontology_type"] == "technique":
                triples.append((e1["entity"], "recommendsTechnique", e2["entity"]))

            # DietaryGuideline → HealthOutcome
            elif e1["ontology_type"] == "dietaryGuideline" and e2["ontology_type"] == "healthOutcome":
                triples.append((e1["entity"], "aimsToImprove", e2["entity"]))

            # DietaryGuideline → EnvironmentImpact
            elif e1["ontology_type"] == "dietaryGuideline" and e2["ontology_type"] == "environmentImpact":
                triples.append((e1["entity"], "guidelineTargetsImpact", e2["entity"]))

            # Technique → EnvironmentImpact
            elif e1["ontology_type"] == "technique" and e2["ontology_type"] == "environmentImpact":
                triples.append((e1["entity"], "affectsImpactCategory", e2["entity"]))
    return triples

# Build RDF triples
for page, page_entities in entities_by_page.items():
    relations = extract_relations(page_entities)
    for subj, pred, obj in relations:
        subj_uri = EX[subj.replace(" ", "_")]
        obj_uri = EX[obj.replace(" ", "_")]
        g.add((subj_uri, EX[pred], obj_uri))
        # store page info as triple-level metadata (comment)
        g.add((subj_uri, RDFS.comment, Literal(f"Found on page {page}")))


# Serialize graph to Turtle
g.serialize("data/triples1.ttl", format="turtle")
print("RDF triples saved to triples.ttl")


RDF triples saved to triples.ttl


## point 5

In [165]:
!pip install rdflib pandas


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [166]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS
import pandas as pd

# Load your triples.ttl file
g = Graph()
g.parse("data/triples1.ttl", format="turtle")

print(f"Loaded {len(g)} triples")



Loaded 4796 triples


In [167]:
EX = Namespace("http://example.org/ex#")
g.bind("ex", EX)

In [168]:
# Example SPARQL query: Count total triples

q1 = """SELECT (COUNT(*) AS ?totalTriples) WHERE { ?s ?p ?o . }"""
for row in g.query(q1):
    print("Total Triples:", row.totalTriples)


Total Triples: 4796


In [169]:
# Example SPARQL query: List all entities and their types

q2 = """
SELECT ?entity ?nutrient
WHERE {
  ?entity ex:hasNutrient ?nutrient .
}
LIMIT 20
"""
pd.DataFrame(g.query(q2), columns=["Entity", "Nutrient"])


,Entity,Nutrient
0,http://example.org/food#alcohol,http://example.org/food#calcium
1,http://example.org/food#barley,http://example.org/food#calcium
2,http://example.org/food#berries,http://example.org/food#calcium
3,http://example.org/food#beverages,http://example.org/food#calcium
4,http://example.org/food#cereals,http://example.org/food#calcium
5,http://example.org/food#dairy,http://example.org/food#calcium
6,http://example.org/food#eggs,http://example.org/food#calcium
7,http://example.org/food#fats,http://example.org/food#calcium
8,http://example.org/food#fish,http://example.org/food#calcium
9,http://example.org/food#fruits,http://example.org/food#calcium


In [170]:
q3 = """
SELECT ?entity ?comment
WHERE {
  ?entity rdfs:comment ?comment .
}
ORDER BY ?comment
LIMIT 5
"""

results = g.query(q3)
df = pd.DataFrame(results, columns=["Entity", "Comment"])

# Optionally extract numeric page number from the comment text
df["Page"] = df["Comment"].str.extract(r"(\d+)")
df.drop(columns=["Comment"], inplace=True)
df


,Entity,Page
0,http://example.org/food#poultry,10
1,http://example.org/food#processed_meat,10
2,http://example.org/food#red_meat,10
3,http://example.org/food#alcohol,102
4,http://example.org/food#berries,102


## Point 6

In [171]:
from rdflib import Graph, Namespace, Literal
import pandas as pd, json

g = Graph()
g.parse("data/triples1.ttl", format="turtle")
EX = Namespace("http://example.org/ex#")
g.bind("ex", EX)
print(f"Loaded {len(g)} triples")


Loaded 4796 triples


In [172]:
print("Sample predicates from your TTL:")
for p in set(g.predicates()):
    print(p)


Sample predicates from your TTL:
http://example.org/food#hasGuideline
http://example.org/food#aimsToImprove
http://example.org/food#usesTechnique
http://example.org/food#affectsRiskOf
http://www.w3.org/2000/01/rdf-schema#comment
http://example.org/food#affectsImpactCategory
http://example.org/food#associatedWithOutcome
http://example.org/food#recommendsTechnique
http://example.org/food#guidelineTargetsImpact
http://example.org/food#hasNutrient
http://example.org/food#hasEnvironmentalImpact


In [173]:
query = """
SELECT ?s ?p ?o ?page
WHERE {
  ?s ?p ?o .
  OPTIONAL { ?s ex:page ?page . }
  FILTER(STRSTARTS(STR(?p), "http://example.org/food#"))
}
"""

rows = []
for s, p, o, page in g.query(query):
    rows.append({
        "subject": str(s).split("#")[-1],
        "predicate": str(p).split("#")[-1],
        "object": str(o).split("#")[-1],
        "page": str(page) if page else None
    })

facts_df = pd.DataFrame(rows)
facts_df.head()


,subject,predicate,object,page
0,include_osteopenia,guidelineTargetsImpact,land_use,None
1,processed_meat,hasGuideline,include_whole,None
2,berries,hasNutrient,protein,None
3,beverages,associatedWithOutcome,blood_pressure,None
4,vegetables,hasNutrient,fibre,None


In [174]:

def make_paraphrases(row):
    s, p, o = row["subject"], row["predicate"], row["object"]

    templates = {
        "hasNutrient": [
            f"{s} contains {o}.",
            f"{o} is a nutrient found in {s}."
        ],
        "usesTechnique": [
            f"{s} is prepared using {o}.",
            f"The preparation of {s} involves {o}."
        ],
        "hasGuideline": [
            f"{s} follows the guideline: {o}.",
            f"The dietary guideline {o} applies to {s}."
        ],
        "recommendsTechnique": [
            f"The guideline {s} recommends {o}.",
            f"{s} suggests using the technique {o}."
        ],
        "aimsToImprove": [
            f"The guideline {s} aims to improve {o}.",
            f"{s} targets the health outcome {o}."
        ],
        "affectsRiskOf": [
            f"{s} affects the risk of {o}.",
            f"Consuming {s} is associated with risk of {o}."
        ],
        "associatedWithOutcome": [
            f"{s} is associated with the health outcome {o}.",
            f"{s} has a relationship with {o}."
        ],
        "hasEnvironmentalImpact": [
            f"{s} has an environmental impact: {o}.",
            f"{s} contributes to {o} impact."
        ],
        "guidelineTargetsImpact": [
            f"The guideline {s} targets environmental impact {o}.",
            f"{s} aims to reduce or manage {o}."
        ],
        "affectsImpactCategory": [
            f"{s} affects the environmental impact category {o}.",
            f"The technique {s} tends to influence {o}."
        ],
    }

    # Fallback for predicates not listed
    result = templates.get(p, [f"{s} {p} {o}."])
    return result if isinstance(result, list) else [str(result)]


# Optional: print first few paraphrases for inspection
for i, row in facts_df.head(3).iterrows():
    print(make_paraphrases(row))


# Apply the paraphrase function
facts_df["paraphrases"] = facts_df.apply(lambda row: make_paraphrases(row), axis=1).astype(object)

# Save facts with paraphrases to JSONL
facts_jsonl_path = "data/facts.jsonl"
with open(facts_jsonl_path, "w", encoding="utf-8") as f:
    for _, row in facts_df.iterrows():
        record = {
            "subject": row["subject"],
            "predicate": row["predicate"],
            "object": row["object"],
            "page": row["page"],
            "paraphrases": row["paraphrases"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Facts with paraphrases saved to {facts_jsonl_path}")


['The guideline include_osteopenia targets environmental impact land_use.', 'include_osteopenia aims to reduce or manage land_use.']
['processed_meat follows the guideline: include_whole.', 'The dietary guideline include_whole applies to processed_meat.']
['berries contains protein.', 'protein is a nutrient found in berries.']
Facts with paraphrases saved to data/facts.jsonl


## Point 7

In [175]:
facts = []
with open("data/facts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        facts.append(json.loads(line))

facts_df = pd.DataFrame(facts)

In [176]:
# return top-k facts supporting a query
def get_grounding(subject=None, predicate=None, object_=None, top_k=3):
    df = facts_df.copy()
    if subject: df = df[df['subject'].str.contains(subject, case=False)]
    if predicate: df = df[df['predicate'].str.contains(predicate, case=False)]
    if object_: df = df[df['object'].str.contains(object_, case=False)]
    df = df.head(top_k)
    return df[['subject','predicate','object','page']].to_dict(orient='records')


In [177]:
def generate_response(instruction_type, **kwargs):
    """
    Returns a text response and grounding facts based on instruction type.
    """
    if instruction_type == "factoid":
        # kwargs: subject, predicate
        # example: subject="Spinach", predicate="hasNutrient"
        grounding = get_grounding(subject=kwargs.get("subject"), predicate=kwargs.get("predicate"), top_k=3)
        if grounding:
            objects = [g['object'] for g in grounding]
            pages = [str(g['page']) for g in grounding if g['page']]
            answer = f"{', '.join(objects)} (Source: p.{', p.'.join(pages)})"
        else:
            answer = "No data found."
            grounding = []
        return answer, grounding

    elif instruction_type == "list":
        # kwargs: predicate, object (filter)
        grounding = get_grounding(predicate=kwargs.get("predicate"), object_=kwargs.get("object_"), top_k=5)
        if grounding:
            objects = [g['subject'] for g in grounding]
            pages = [str(g['page']) for g in grounding if g['page']]
            answer = f"{', '.join(objects)}. (Source: p.{', p.'.join(pages)})"
        else:
            answer = "No data found."
            grounding = []
        return answer, grounding

    # reasoning/constraint can be implemented similarly by filtering on multiple columns


In [178]:
dataset = []

# Example Factoid QA
instruction = "What nutrient is high in eggs?"
output, grounding = generate_response("factoid", subject="eggs", predicate="hasNutrient")
dataset.append({
    "instruction": instruction,
    "input": "",
    "output": output,
    "grounding": grounding
})

# Example List / Compare
instruction2 = "List two ingredients rich in fiber."
output2, grounding2 = generate_response("list", predicate="hasNutrient", object_="fiber")
dataset.append({
    "instruction": instruction2,
    "input": "",
    "output": output2,
    "grounding": grounding2
})

# Save to JSONL
with open("data/instruction_response_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Generated {len(dataset)} instruction-response pairs")


Generated 2 instruction-response pairs


In [179]:
# Generate diverse QA pairs from facts

import json
import random
import pandas as pd
from pathlib import Path

# establish paths
facts_path = Path("data/facts.jsonl")
output_dir = Path("data/train/test/val")
output_dir.mkdir(parents=True, exist_ok=True)

# laad facts
facts = []
with open(facts_path, "r", encoding="utf-8") as f:
    for line in f:
        facts.append(json.loads(line))
facts_df = pd.DataFrame(facts)

print(f"Loaded {len(facts_df)} facts from {facts_path}")


def get_page_ref(pages):
    if not pages:
        return ""
    pages = [str(p) for p in pages if p]
    return f"(Source: p.{', p.'.join(pages)})" if pages else ""

def get_grounding(subset):
    grounding = []
    for _, row in subset.iterrows():
        grounding.append({
            "s": row["subject"],
            "p": row["predicate"],
            "o": row["object"],
            "page": row["page"]
        })
    return grounding


# define templates for different types of questions
TEMPLATES = {
    "hasNutrient": {
        "factoid": [
            "What nutrient is found in {s}?",
            "Which nutrient does {s} provide?",
            "What is the main nutrient in {s}?",
            "What does {s} contain that supports health?",
            "{s} is rich in which nutrient?"
        ],
        "list": [
            "List foods rich in {o}.",
            "Which ingredients are good sources of {o}?",
            "Give examples of foods containing {o}.",
            "Name three foods that are high in {o}."
        ],
        "constraint": [
            "Give two ingredients that contain {o}.",
            "Find vegan foods rich in {o}.",
            "List foods with high {o} content under 200 kcal per 100g."
        ]
    },
    "associatedWithOutcome": {
        "factoid": [
            "What health outcome is {s} associated with?",
            "How does consuming {s} affect health?",
            "{s} is linked to which health condition?",
            "What condition may be influenced by {s}?"
        ],
        "list": [
            "List foods associated with reduced risk of {o}.",
            "Which foods promote {o}?",
            "Name foods linked to {o}."
        ]
    },
    "affectsRiskOf": {
        "factoid": [
            "What health outcome does {s} influence?",
            "Which disease risk is affected by {s}?",
            "How does {s} affect the risk of {o}?",
            "What role does {s} play in preventing {o}?"
        ]
    },
    "usesTechnique": {
        "factoid": [
            "What technique is used to prepare {s}?",
            "How is {s} typically cooked or processed?",
            "What preparation method applies to {s}?"
        ]
    },
    "affectsImpactCategory": {
        "factoid": [
            "What environmental impact is influenced by {s}?",
            "Which sustainability impact is affected by {s}?",
            "How does {s} affect environmental footprint?"
        ],
        "reasoning": [
            "If a recipe uses {s}, what environmental impact may increase?",
            "When {s} is used, which environmental category might be affected?"
        ]
    },
    "hasGuideline": {
        "factoid": [
            "What dietary guideline applies to {s}?",
            "What recommendation is given regarding {s}?",
            "What official advice mentions {s}?"
        ]
    },
    "aimsToImprove": {
        "factoid": [
            "What health outcome does {s} aim to improve?",
            "What benefit does {s} target?",
            "Which condition is addressed by {s}?"
        ]
    },
    "guidelineTargetsImpact": {
        "factoid": [
            "What environmental impact does {s} aim to reduce?",
            "Which sustainability metric is targeted by {s}?"
        ]
    }
}



# generate functions of different type
def generate_factoid_qa(fact):
    # TYPE 1: Factoid-style questions
    s, p, o, page = fact["subject"], fact["predicate"], fact["object"], fact["page"]

    templates = TEMPLATES.get(p, {}).get("factoid", [])
    if not templates:
        instruction = f"What is the relationship between {s.replace('_', ' ')} and {o.replace('_', ' ')}?"
        output = f"{s.replace('_', ' ')} {p.replace('_', ' ')} {o.replace('_', ' ')}. {get_page_ref([page])}"
    else:
        instruction = random.choice(templates).format(s=s.replace("_", " "), o=o.replace("_", " "))
        output = f"{s.replace('_', ' ')} {p.replace('_', ' ')} {o.replace('_', ' ')}. {get_page_ref([page])}"

    grounding = [fact.to_dict()]
    return {"instruction": instruction.strip(), "input": "", "output": output.strip(), "grounding": grounding}


def generate_list_compare(df):
    # TYPE 2: List/Compare-style questions
    nutrient_rows = df[df["predicate"] == "hasNutrient"]
    if len(nutrient_rows) < 3:
        return None

    nutrient = random.choice(nutrient_rows["object"].unique().tolist())
    subset = nutrient_rows[nutrient_rows["object"] == nutrient].sample(min(4, len(nutrient_rows)))
    ingredients = [s.replace('_', ' ') for s in subset["subject"].tolist()]
    pages = subset["page"].tolist()

    templates = TEMPLATES["hasNutrient"]["list"]
    instruction = random.choice(templates).format(o=nutrient.replace("_", " "))
    output = "; ".join(ingredients) + f". {get_page_ref(pages)}"
    grounding = get_grounding(subset)
    return {"instruction": instruction.strip(), "input": "", "output": output.strip(), "grounding": grounding}


def generate_reasoning(df):
    # TYPE 3: Reasoning-style questions
    tech_rows = df[df["predicate"] == "affectsImpactCategory"]
    if len(tech_rows) == 0:
        return None

    row = tech_rows.sample(1).iloc[0]
    s, o, page = row["subject"], row["object"], row["page"]
    templates = TEMPLATES["affectsImpactCategory"]["reasoning"]
    instruction = random.choice(templates).format(s=s.replace("_", " "), o=o.replace("_", " "))
    output = f"{s.replace('_', ' ')} affects {o.replace('_', ' ')} impact. {get_page_ref([page])}"
    grounding = [row.to_dict()]
    return {"instruction": instruction.strip(), "input": "", "output": output.strip(), "grounding": grounding}


def generate_constraint_query(df):
    # TYPE 4: Constraint-style questions
    nutrient_rows = df[df["predicate"] == "hasNutrient"]
    if len(nutrient_rows) == 0:
        return None

    nutrient = random.choice(nutrient_rows["object"].unique().tolist())
    subset = nutrient_rows[nutrient_rows["object"] == nutrient].sample(min(3, len(nutrient_rows)))
    ingredients = [s.replace('_', ' ') for s in subset["subject"].tolist()]
    pages = subset["page"].tolist()

    templates = TEMPLATES["hasNutrient"]["constraint"]
    instruction = random.choice(templates).format(o=nutrient.replace("_", " "))
    output = "; ".join(ingredients) + f". {get_page_ref(pages)}"
    grounding = get_grounding(subset)
    return {"instruction": instruction.strip(), "input": "", "output": output.strip(), "grounding": grounding}


# create the dataset
dataset = []

# one QA per fact
for _, fact in facts_df.iterrows():
    dataset.append(generate_factoid_qa(fact))

# extra random examples
for _ in range(40):
    q = generate_list_compare(facts_df)
    if q: dataset.append(q)

for _ in range(20):
    q = generate_reasoning(facts_df)
    if q: dataset.append(q)

for _ in range(20):
    q = generate_constraint_query(facts_df)
    if q: dataset.append(q)

dataset = [d for d in dataset if d]
print(f"Generated {len(dataset)} instruction–response pairs.")

# split in train, test and validation
# after shuffling 80% train, 10% val, 10% test
random.shuffle(dataset)
n = len(dataset)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

splits = {
    "train": dataset[:train_end],
    "val": dataset[train_end:val_end],
    "test": dataset[val_end:]
}

for split_name, data in splits.items():
    out_path = output_dir / f"{split_name}_instructions.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for record in data:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(data)} {split_name} examples → {out_path}")


Loaded 4218 facts from data/facts.jsonl
Generated 4298 instruction–response pairs.
Saved 3438 train examples → data/train/test/val/train_instructions.jsonl
Saved 430 val examples → data/train/test/val/val_instructions.jsonl
Saved 430 test examples → data/train/test/val/test_instructions.jsonl


## Point 7

In [180]:
# comparing two ways of answering questions using your knowledge graph (KG) and/or an LLM

# idea:
# Zero-shot: “I’m asking GPT what it remembers about nutrition from training data.”
# RAG-KG: “I’m giving GPT the verified facts from the PDF and asking it to answer based only on that.”

